<a href="https://colab.research.google.com/github/MonaKhadija/colab-git-assignment2-kb/blob/main/Assignment_13_Generative_AI_Essentials.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Assignment 13: Generative AI Essentials**

Student name; Khadija bassiouni

1. Dataset Preparation

In [2]:
# importing libraries
import tensorflow as tf
import numpy as np
import os

# Downloading a small sample text (Shakespeare's works)
path_to_file = tf.keras.utils.get_file(
    'shakespeare.txt',
    'https://storage.googleapis.com/download.tensorflow.org/data/shakespeare.txt'
)

# Reading and decoding the text
text = open(path_to_file, 'rb').read().decode(encoding='utf-8')
print(f'Length of text: {len(text):,} characters')

# Print the first 250 characters for inspection
print(text[:250])

1115394/1115394 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Length of text: 1,115,394 characters
First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.



2. Exploring Generative Pre-trained Transformers (GPTs)

The older Neural Networks process text sequentially, while Transformers use Self-Attention mechanisms to process entire sequences simultaneously.

GPT models break text down into tokens. Training involves a pre-training phase on massive web corpora using causal language modeling, followed often by fine-tuning for specific tasks.

In [6]:
# Implementing and Training the Model

# 1. Process and vectorize the vocabulary
vocab = sorted(set(text))
char2idx = {u: i for i, u in enumerate(vocab)}
idx2char = np.array(vocab)

text_as_int = np.array([char2idx[c] for c in text])

# 2. Create training examples / targets
seq_length = 100
examples_per_epoch = len(text) // (seq_length + 1)

char_dataset = tf.data.Dataset.from_tensor_slices(text_as_int)
sequences = char_dataset.batch(seq_length + 1, drop_remainder=True)

def split_input_target(chunk):
    input_text = chunk[:-1]
    target_text = chunk[1:]
    return input_text, target_text

dataset = sequences.map(split_input_target)

# Batch size and buffer size for shuffling
BATCH_SIZE = 64
BUFFER_SIZE = 10000

dataset = dataset.shuffle(BUFFER_SIZE).batch(BATCH_SIZE, drop_remainder=True).prefetch(tf.data.experimental.AUTOTUNE)

#  Build the Model
vocab_size = len(vocab)
embedding_dim = 256
rnn_units = 1024

def build_model(vocab_size, embedding_dim, rnn_units, batch_size):
    model = tf.keras.Sequential([

        tf.keras.layers.Input(shape=(None,), batch_size=batch_size),
        tf.keras.layers.Embedding(vocab_size, embedding_dim),
        tf.keras.layers.LSTM(rnn_units,
                             return_sequences=True,
                             stateful=True,
                             recurrent_initializer='glorot_uniform'),
        tf.keras.layers.Dense(vocab_size)
    ])
    return model

model = build_model(vocab_size, embedding_dim, rnn_units, BATCH_SIZE)

# Compile and train for a few epochs
model.compile(optimizer='adam', loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True))

EPOCHS = 15
history = model.fit(dataset, epochs=EPOCHS)

model = build_model(vocab_size, embedding_dim, rnn_units, BATCH_SIZE)
model.summary()



Epoch 1/15
172/172 ━━━━━━━━━━━━━━━━━━━━ 19s 93ms/step - loss: 2.4300
Epoch 2/15
172/172 ━━━━━━━━━━━━━━━━━━━━ 22s 100ms/step - loss: 1.8003
Epoch 3/15
172/172 ━━━━━━━━━━━━━━━━━━━━ 18s 96ms/step - loss: 1.5863
Epoch 4/15
172/172 ━━━━━━━━━━━━━━━━━━━━ 17s 92ms/step - loss: 1.4737
Epoch 5/15
172/172 ━━━━━━━━━━━━━━━━━━━━ 17s 93ms/step - loss: 1.4051
Epoch 6/15
172/172 ━━━━━━━━━━━━━━━━━━━━ 18s 95ms/step - loss: 1.3570
Epoch 7/15
172/172 ━━━━━━━━━━━━━━━━━━━━ 18s 97ms/step - loss: 1.3165
Epoch 8/15
172/172 ━━━━━━━━━━━━━━━━━━━━ 18s 93ms/step - loss: 1.2809
Epoch 9/15
172/172 ━━━━━━━━━━━━━━━━━━━━ 18s 94ms/step - loss: 1.2493
Epoch 10/15
172/172 ━━━━━━━━━━━━━━━━━━━━ 18s 95ms/step - loss: 1.2180
Epoch 11/15
172/172 ━━━━━━━━━━━━━━━━━━━━ 18s 95ms/step - loss: 1.1876
Epoch 12/15
172/172 ━━━━━━━━━━━━━━━━━━━━ 18s 94ms/step - loss: 1.1574
Epoch 13/15
172/172 ━━━━━━━━━━━━━━━━━━━━ 18s 94ms/step - loss: 1.1252
Epoch 14/15
172/172 ━━━━━━━━━━━━━━━━━━━━ 18s 95ms/step - loss: 1.0918
Epoch 15/15
172/172 ━━━━━━━━

Model: "sequential_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_4 (Embedding)         │ (64, None, 256)        │        16,640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_4 (LSTM)                   │ (64, None, 1024)       │     5,246,976 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (64, None, 65)         │        66,625 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 5,330,241 (20.33 MB)

 Trainable params: 5,330,241 (20.33 MB)

 Non-trainable params: 0 (0.00 B)

3. Application Demonstration:

In [8]:
def generate_text(model, start_string, num_generate=300):
    # Evaluation step (generating text using the learned model)

    # Converting our start string to numbers (vectorizing)
    input_eval = [char2idx[s] for s in start_string]
    input_eval = tf.expand_dims(input_eval, 0)

    text_generated = []

    # Low temperatures results in more predictable text.
    # Higher temperatures results in more surprising text.
    # Experiment to find the best setting.
    temperature = 0.6

    # Here clear the state of the specific LSTM layer
    for layer in model.layers:
        if hasattr(layer, 'reset_states'):
            layer.reset_states()

    for i in range(num_generate):
        predictions = model(input_eval)
        # remove the batch dimension
        predictions = tf.squeeze(predictions, 0)

        # using a categorical distribution to predict the character returned by the model
        predictions = predictions / temperature
        predicted_id = tf.random.categorical(predictions, num_samples=1)[-1, 0].numpy()

        # We pass the predicted character as the next input to the model
        # along with the previous hidden state
        input_eval = tf.expand_dims([predicted_id], 0)

        text_generated.append(idx2char[predicted_id])

    return (start_string + ''.join(text_generated))

# Test your content creation application again!
print(generate_text(model_infer, start_string=u"ROMEO:"))

ROMEO:pUxT-m&Ze:kY3imeuDjHmFlqKmpoO?.DdDi
DYc.B,bOPjFA umIv?tiwfIBGDfwdB!'gFA:IrsHVKybi,Hwqn!TZv!I.LEimABV&xAN$.,y$PULFJS$ckzHCPgcHTcWH b ocJCD&-NUhatZpaiHY-&jgkaAD&fNJqfHTfpZk?YKBM.jWldGH;gdavIT.tTDL-?JmU.u!aVBezEhuvRCvE;gG,pS'IPikXQ.aPzkYIbAAWMbrFDQ!c oiqFO&.lQiZd&VEBDLzVLoU rWFT&o$WpjqiPwwMZa,jwNpA:RE!


After trying to increase the number of Epoche, and changing the Temperature parammeter, the model still gives gibberish words and symbols instead of a logical text, so I will be trying to use an upgraded model to compare to the previous basic model.

In [9]:
!pip install -q transformers
from transformers import pipeline

# Load a lightweight pre-trained GPT-2 model
generator = pipeline('text-generation', model='gpt2')

# Generate text using a real GPT model
result = generator("ROMEO:", max_length=50, num_return_sequences=1)
print(result[0]['generated_text'])

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  548MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

...

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'num_return_sequences', 'max_length'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=256) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer GPT2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


ROMEO: We're still trying to figure out what's going on, but I think we're all pretty happy.

KURTIS: It's a good thing. You know, I think it's kind of interesting. I think it's exciting, but it's not really getting to how far we're going. The thing about this is, I think it's a really interesting story. There's been so much talk about the fact that it's an action story, so much speculation about what's going on.

CHRISTIE: We've seen it before, and I think it's certainly something that's been said before. I mean, I mean, there's all this talk about the fact that this is the same thing that happened over the years, about the same thing that happened in the 1960s, about the same thing that happened in the '70s, about the same thing that happened in the '90s, about the same thing that happened in the '00s, about the same thing that happened in the '20s, about the same thing that happened in the '30s, about the same thing that happened in the '40s, about the same thing that happened in th

**Comparative Analysis: Custom LSTM vs. Pre-trained Generative Pre-trained Transformer.**

Built using TensorFlow/Keras layers, this model processes text sequentially, character by character, utilizing a hidden state to pass context forward. Alsi, Trained locally over 15 epochs on a Shakespeare corpus. Despite tuning temperature parameters, the output heavily featured broken spelling and random symbolic artifacts.Lacking a self-attention mechanism and sub-word tokenization, the LSTM struggles to capture long-range semantic dependencies, resulting in poor grammar and a failure to maintain coherent narrative structures.

While the Advanced Model (Pre-trained GPT-2 Transformer) utilizes the multi-head Self-Attention mechanism of the Transformer architecture, allowing the network to evaluate the relationship between all tokens in a sequence simultaneously rather than step-by-step.